In [2]:
from pyspark import SparkConf, SparkContext

conf = (SparkConf()
        .setAppName("List_Of_Countries")
        .setMaster("local[*]")    
)
sc = SparkContext(conf=conf)
path_file = "List_of_Countries.txt"

### RDD
#### 1. Загрузите файл List of Countries.txt в RDD. 

In [3]:
raw_rdd = sc.textFile(path_file)

#### 2. Определите количество стран для каждой первой буквы их названия (сколько стран начинается на 'A', сколько на 'B' и так далее). Выведите результат в алфавитном порядке по букве.

In [5]:
first_letter_contry = raw_rdd.map(lambda x: (x[0], 1))
cnt_flc = first_letter_contry.reduceByKey(lambda a, b: a + b)
print(f"Кол-во стран для каждой первой буквы:")
for first_letter, cnt in cnt_flc.sortByKey().collect():
    print(f"{first_letter}: {cnt}")

Кол-во стран для каждой первой буквы:
A: 10
B: 16
C: 18
D: 5
E: 8
F: 4
G: 12
H: 5
I: 9
J: 3
K: 4
L: 9
M: 18
N: 9
O: 1
P: 7
Q: 1
R: 4
S: 26
T: 11
U: 7
V: 2
W: 1
Y: 1
Z: 2


#### 3. Найдите 5 самых длинных названий стран в списке. Если есть несколько стран с одинаковой максимальной длиной, выведите их все. 

In [11]:
# Пробел в названии тоже будем считать
# дополнительно чистим от возможных пробелов в начале и в конце строк
clean_rdd = raw_rdd.map(lambda line: line.strip())

length_rdd = clean_rdd.map(lambda line: (line, len(line)))
top_5_length = length_rdd.top(5, key = lambda x: x[1])
#print(*top_5_length, sep="\n")
print("5 самых длинных названий стран:")
for country, len_country in top_5_length:
    print(f"{country}: {len_country}")

5 самых длинных названий стран:
Saint Vincent and the Grenadines: 32
Central African Republic: 24
Bosnia and Herzegovina: 22
Saint Kitts and Nevis: 21
United Arab Emirates: 20


#### 4. Найдите 5 самых коротких названий стран в списке. Аналогично, если есть несколько с одинаковой минимальной длиной, выведите их все.

In [13]:
# возьмем уже готовый rdd и просто отсортируем в обратном порядке
top_5_length_short = length_rdd.top(5, key = lambda x: -x[1])
print("5 самых коротких названий стран:")
for country, len_country in top_5_length_short:
    print(f"{country}: {len_country}")

5 самых коротких названий стран:
Chad: 4
Cuba: 4
Iran: 4
Iraq: 4
Laos: 4


#### 5. Рассчитайте среднюю длину названия стран в списке. Округлите результат до одного знаков после запятой.

In [21]:
total_sum_countries = length_rdd.map(lambda x: x[1]).reduce(lambda a, b: a + b)
cnt_countries = clean_rdd.count()
print(f"Средняя длина строки в названии стран: {round(total_sum_countries/cnt_countries, 1)} ")

Средняя длина строки в названии стран: 8.5 


#### 6. Найдите все страны, названия которых состоят из двух и более слов. 

In [27]:
rdd_with_many_words = clean_rdd.filter(lambda x: " " in x)
print(rdd_with_many_words.collect())
print(rdd_with_many_words.count())

['Antigua and Barbuda', 'Bosnia and Herzegovina', 'Burkina Faso', 'Cabo Verde', 'Central African Republic', 'Channel Islands', 'Costa Rica', "Côte d'Ivoire", 'Czech Republic', 'Dominican Republic', 'DR Congo', 'El Salvador', 'Equatorial Guinea', 'Faeroe Islands', 'French Guiana', 'Holy See', 'Hong Kong', 'Isle of Man', 'North Korea', 'North Macedonia', 'Saint Helena', 'Saint Kitts and Nevis', 'Saint Lucia', 'Saint Vincent and the Grenadines', 'San Marino', 'Sao Tome & Principe', 'Saudi Arabia', 'Sierra Leone', 'South Africa', 'South Korea', 'South Sudan', 'Sri Lanka', 'State of Palestine', 'The Bahamas', 'Trinidad and Tobago', 'United Arab Emirates', 'United Kingdom', 'United States', 'Western Sahara']
39


In [28]:
sc.stop()

### DataFrame
#### 1. Загрузите файл List of Countries.txt

In [30]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, trim, sum, min, max, length, avg, round, count, substring
from pyspark.sql.types import StructType, StructField, StringType

In [41]:
# Инициализация сессии
spark = SparkSession.builder \
        .appName("List_Of_Countries") \
        .master("local[*]") \
        .getOrCreate()

# Загрузка файла с названиями стран
file_path = "List_of_Countries.txt"
file_schema = StructType([
    StructField("country", StringType(), True)
])

df_raw_countries = spark.read.csv(
    file_path,
    header=False,
    schema=file_schema
)

# сразу немного почистим
df_clean_countries = df_raw_countries.select(trim(col("country")).alias("country"))
df_clean_countries.show(5)

+-----------+
|    country|
+-----------+
|Afghanistan|
|    Albania|
|    Algeria|
|    Andorra|
|     Angola|
+-----------+
only showing top 5 rows



#### 2. Определите количество стран для каждой первой буквы их названия (сколько стран начинается на 'A', сколько на 'B' и так далее). Выведите результат в алфавитном порядке по букве.

In [47]:
df_first_letter_country = df_clean_countries.groupBy(substring(col("country"), 1, 1).alias("first_letter")).agg(
    count("*").alias("cnt_first_letter")
).orderBy(col("first_letter"))
df_first_letter_country.show()

+------------+----------------+
|first_letter|cnt_first_letter|
+------------+----------------+
|           A|              10|
|           B|              16|
|           C|              18|
|           D|               5|
|           E|               8|
|           F|               4|
|           G|              12|
|           H|               5|
|           I|               9|
|           J|               3|
|           K|               4|
|           L|               9|
|           M|              18|
|           N|               9|
|           O|               1|
|           P|               7|
|           Q|               1|
|           R|               4|
|           S|              26|
|           T|              11|
+------------+----------------+
only showing top 20 rows



#### 3. Найдите 5 самых длинных названий стран в списке. Если есть несколько стран с одинаковой максимальной длиной, выведите их все. 

In [48]:
top_5_longest_names = df_clean_countries.withColumn(
    "len_country",
    length(col("country"))
).orderBy(col("len_country").desc())

top_5_longest_names.show(5)

+--------------------+-----------+
|             country|len_country|
+--------------------+-----------+
|Saint Vincent and...|         32|
|Central African R...|         24|
|Bosnia and Herzeg...|         22|
|Saint Kitts and N...|         21|
|United Arab Emirates|         20|
+--------------------+-----------+
only showing top 5 rows



#### 4. Найдите 5 самых коротких названий стран в списке. Аналогично, если есть несколько с одинаковой минимальной длиной, выведите их все.

In [49]:
top_5_shortest_names = top_5_longest_names.orderBy(col("len_country"))
top_5_shortest_names.show(5)

+-------+-----------+
|country|len_country|
+-------+-----------+
|   Iran|          4|
|   Peru|          4|
|   Iraq|          4|
|   Cuba|          4|
|   Laos|          4|
+-------+-----------+
only showing top 5 rows



#### 5. Рассчитайте среднюю длину названия стран в списке. Округлите результат до одного знаков после запятой.

In [58]:
avg_name_length = top_5_longest_names.agg(
    round(avg(col("len_country")), 1).alias("avg_len")
).collect()[0][0]

print(f"Средняя длина названия стран: {avg_name_length}")

Средняя длина названия стран: 8.5


#### 6. Найдите все страны, названия которых состоят из двух и более слов. 

In [61]:
# rdd_with_many_words = clean_rdd.filter(lambda x: " " in x)
df_countries_with_many_words = df_clean_countries.filter(col("country").contains(' '))
df_countries_with_many_words.show()
print(df_countries_with_many_words.count())

+--------------------+
|             country|
+--------------------+
| Antigua and Barbuda|
|Bosnia and Herzeg...|
|        Burkina Faso|
|          Cabo Verde|
|Central African R...|
|     Channel Islands|
|          Costa Rica|
|       Côte d'Ivoire|
|      Czech Republic|
|  Dominican Republic|
|            DR Congo|
|         El Salvador|
|   Equatorial Guinea|
|      Faeroe Islands|
|       French Guiana|
|            Holy See|
|           Hong Kong|
|         Isle of Man|
|         North Korea|
|     North Macedonia|
+--------------------+
only showing top 20 rows

39


In [62]:
spark.stop()